# Qdrant basics
This notebook is a basic example of how to use Qdrant, including:
- Creating a collection
- Deleting a collection
- Adding points (vectors) to the collection
- Deleting points from the collection
- Searching for points in the collection

Let's import the necessary libraries:

In [1]:
from random import random
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

# Staring with Qdrant
To setup Qdrant vector database, follow the instructions from the official documentation:
https://qdrant.tech/documentation/quickstart/

# Initialize Qdrant client

In [2]:
client = QdrantClient(url="http://localhost:6333")

# Add collection
Points (vectors) are stored in collections. Here is how to add a collection.

We need to pass two parameters:
- collection_name - name of the collection
- vectors_config - configuration of the vectors (size and distance calculation method)

In [3]:
dimensions = 1536 # Make sure the dimension count matches the embedding model you are using, is this case ada-002

client.create_collection(
    collection_name="my_first_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

client.get_collection("my_first_collection")

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, vectors_count=None, indexed_vectors_count=0, points_count=0, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None), pa

# Preview collection
To preview your collection, you can go to this address:

http://localhost:6333/dashboard#/collections

or directly to the collection:

http://localhost:6333/dashboard#/collections/my_first_collection

# Removing collection
Let's create a second collection and then remove it.

In [4]:
client.create_collection(
    collection_name="my_temporary_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

client.get_collection("my_temporary_collection")

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, vectors_count=None, indexed_vectors_count=0, points_count=0, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None), pa

Now lets remove this collection.

In [5]:
client.delete_collection("my_temporary_collection")

try:
    client.get_collection("my_temporary_collection")
except Exception as e:
    print(e)


Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `my_temporary_collection` doesn\'t exist!"},"time":0.000040292}'


# Adding points
Now, let's add some points to the collection.

First we need to create a vector for the point. We will use a mock vector for this example.

In [6]:
vector = [random() for _ in range(dimensions)] # This will create a vector with 1536 random values between 0 and 1

# Let's call the upsert method to add the point to the collection
client.upsert(
    collection_name="my_first_collection",
    wait=True,
    points=[
        PointStruct(id=1, vector=vector),
    ],
)

# We can retrieve the point by its id
client.retrieve(
    collection_name="my_first_collection",
    ids=[1],
    with_vectors=False, # We don't need to retrieve the vector for this example
)

[Record(id=1, payload={}, vector=None, shard_key=None)]

We can also include additional information in the point as metadata.

This metadata can be used for filtering and grouping points.

In [7]:
metadata = {
    "city": "Kraków",
    "location": "Prompt Haus",
}

# Let's add the point to the collection
client.upsert(
    collection_name="my_first_collection",
    wait=True,
    points=[
        PointStruct(id=2, vector=vector, payload=metadata),
    ],
)

# Let's see the point we just added - you can see the metadata
client.retrieve(
    collection_name="my_first_collection",
    ids=[2],
    with_vectors=False, # We don't need to retrieve the vector for this example
)

[Record(id=2, payload={'city': 'Kraków', 'location': 'Prompt Haus'}, vector=None, shard_key=None)]

# Search for points
Now, let's search for points in the collection.

First we need to add some more points to the collection.


In [8]:
mock_vectors = [
    [random() for _ in range(dimensions)],
    [random() for _ in range(dimensions)],
    [random() for _ in range(dimensions)],
    [random() for _ in range(dimensions)],
    [random() for _ in range(dimensions)],
]

for i, vector in enumerate(mock_vectors):
    metadata = {
        "test_id": i,
    }

    client.upsert(
        collection_name="my_first_collection",
        wait=True,
        points=[
            PointStruct(id=i, vector=vector, payload=metadata),
        ],
    )

# Let's see how many points we have in our collection now
client.get_collection("my_first_collection")

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, vectors_count=None, indexed_vectors_count=0, points_count=5, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None), pa

Let's create a vector for the search query and search for points in the collection.

Result should be a list of points with the highest cosine similarity to the search vector.

You can also see the metadata of the points in the result.

In [9]:
mock_vector_for_search = [random() for _ in range(dimensions)]

# Let's search for points in the collection
client.search(
    collection_name="my_first_collection",
    query_vector=mock_vector_for_search,
    limit=5,
    with_vectors=False, # We don't need to retrieve the vector for this example
)

[ScoredPoint(id=1, version=3, score=0.7587888, payload={'test_id': 1}, vector=None, shard_key=None),
 ScoredPoint(id=3, version=5, score=0.7520355, payload={'test_id': 3}, vector=None, shard_key=None),
 ScoredPoint(id=4, version=6, score=0.74587744, payload={'test_id': 4}, vector=None, shard_key=None),
 ScoredPoint(id=0, version=2, score=0.7428986, payload={'test_id': 0}, vector=None, shard_key=None),
 ScoredPoint(id=2, version=4, score=0.7417184, payload={'test_id': 2}, vector=None, shard_key=None)]